# Lezione 3 · Visione artificiale II: architetture avanzate

**Corso:** Advanced Deep Learning (apprendimento profondo avanzato)

Benvenuto in questa lezione dal vivo. Vedrai, con codice funzionante, perché le reti molto profonde hanno bisogno di una connessione residua e come riutilizzare una rete neurale convoluzionale (CNN) già addestrata invece di partire da zero.

## Obiettivi di apprendimento

Al termine di questa lezione sarai in grado di:

- Spiegare perché impilare troppi livelli in una rete profonda ne degrada le prestazioni
- Implementare un blocco residuo in PyTorch e verificarne il passo in avanti (*forward pass*)
- Caricare una CNN preaddestrata con `torchvision` e usarla per classificare un'immagine
- Applicare il trasferimento di apprendimento (*transfer learning*): congelare i livelli e adattare l'ultimo livello a un nuovo numero di classi

## Il problema delle reti molto profonde

Quando impili più livelli in una rete neurale convoluzionale (CNN), ti aspetti che le prestazioni migliorino. Ma oltre una certa profondità accade il contrario: l'errore di addestramento torna a salire. Non si tratta di *overfitting*: è un problema di ottimizzazione noto come degradazione.

La causa è la scomparsa del gradiente (*vanishing gradient*). Durante l'addestramento il gradiente si propaga all'indietro, livello dopo livello, tramite la regola della catena. Se in ogni livello il fattore è minore di uno, il prodotto di molti fattori piccoli tende a zero molto prima di raggiungere i primi livelli. Questi livelli smettono quindi di apprendere, anche se hanno la capacità per farlo.

Ti serve un modo per garantire che il gradiente arrivi con forza sufficiente fino ai primi livelli, indipendentemente da quanti livelli abbia la rete. Questa è la motivazione alla base della connessione residua.

## La connessione residua

Immagina un percorso escursionistico con molte tappe consecutive, tutte in salita. Se perdi un po' di energia a ogni tappa, arrivato alla tappa 50 potresti non avere più forze per continuare. È quello che succede al gradiente in una rete molto profonda durante l'addestramento.

Ora immagina che, accanto al sentiero principale, esista una scorciatoia che scende direttamente dall'inizio fino a ogni rifugio. Non sostituisce il sentiero principale, ma ti garantisce sempre una via diretta.

È questa l'idea di ResNet (Residual Network, "rete residua"): invece di chiedere a un blocco di livelli di apprendere una trasformazione completa H(x), gli si chiede di apprendere solo il residuo F(x) = H(x) − x. L'output effettivo del blocco è:

y = F(x) + x

Se la scelta migliore per un blocco è lasciar passare l'input senza modifiche, la rete deve solo imparare a portare F(x) vicino a zero, un compito molto più semplice che apprendere un'identità esatta con livelli non lineari. E durante la retropropagazione il gradiente dispone sempre di una via diretta attraverso la somma, oltre a quella che attraversa le convoluzioni.

Verifica questa idea con un blocco residuo costruito a mano.

In [ ]:
import torch
import torch.nn as nn


class BloqueResidual(nn.Module):
    """Bloque residual: dos convoluciones 3x3 y un atajo (shortcut)
    que suma la entrada a la salida antes de la última activación."""

    def __init__(self, canales_entrada, canales_salida, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(canales_entrada, canales_salida, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(canales_salida)
        self.conv2 = nn.Conv2d(canales_salida, canales_salida, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(canales_salida)
        self.relu = nn.ReLU(inplace=True)

        # Si el bloque cambia la resolución o el número de canales,
        # el atajo necesita una proyección 1x1 para poder sumarse a F(x)
        self.atajo = nn.Sequential()
        if stride != 1 or canales_entrada != canales_salida:
            self.atajo = nn.Sequential(
                nn.Conv2d(canales_entrada, canales_salida, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(canales_salida),
            )

    def forward(self, x):
        identidad = self.atajo(x)                 # atajo: y = F(x) + x
        salida = self.relu(self.bn1(self.conv1(x)))
        salida = self.bn2(self.conv2(salida))
        salida = salida + identidad                # aquí ocurre la conexión residual
        return self.relu(salida)


# Prueba rápida: lote de 2 imágenes, 3 canales, 32x32 píxeles
entrada_prueba = torch.randn(2, 3, 32, 32)
bloque = BloqueResidual(canales_entrada=3, canales_salida=16, stride=2)
salida_prueba = bloque(entrada_prueba)

print("Forma de entrada:", entrada_prueba.shape)
print("Forma de salida: ", salida_prueba.shape)

## Usa un modello preaddestrato

Addestrare una ResNet da zero richiede milioni di immagini e molte ore di calcolo. Non devi ripetere questo lavoro: `torchvision` ti dà accesso diretto a reti già addestrate su ImageNet, un benchmark di oltre un milione di immagini suddivise in 1000 categorie.

Carica una ResNet-18 preaddestrata, applicala a un'immagine reale scaricata da internet e osserva le sue cinque previsioni con probabilità più alta (accuratezza top-5).

In [ ]:
import urllib.request

import torch
from PIL import Image
from torchvision import models, transforms

# Carga la ResNet-18 con los pesos entrenados sobre ImageNet
modelo = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
modelo.eval()  # desactiva el dropout y fija las estadísticas de BatchNorm

# Descarga una imagen de ejemplo y la lista de las 1000 clases de ImageNet
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
    "imagen_ejemplo.jpg",
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
    "clases_imagenet.txt",
)
with open("clases_imagenet.txt") as archivo:
    categorias = [linea.strip() for linea in archivo.readlines()]

imagen = Image.open("imagen_ejemplo.jpg").convert("RGB")
imagen

In [ ]:
preprocesar = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # medias de ImageNet
                          std=[0.229, 0.224, 0.225]),
])
entrada = preprocesar(imagen).unsqueeze(0)  # añade la dimensión de lote

with torch.no_grad():
    salida = modelo(entrada)
probabilidades = torch.nn.functional.softmax(salida[0], dim=0)

top5_prob, top5_idx = torch.topk(probabilidades, 5)
for probabilidad, indice in zip(top5_prob, top5_idx):
    print(f"{categorias[indice]}: {probabilidad.item() * 100:.1f} %")

## Trasferimento di apprendimento

Il trasferimento di apprendimento (*transfer learning*) riutilizza ciò che un modello ha già appreso su un compito di origine per accelerare l'apprendimento su un compito di destinazione. La premessa è semplice: molte caratteristiche visive (bordi, texture, forme) servono a più di un compito, quindi non serve riapprenderle ogni volta.

La procedura abituale prevede due passi:

- **Congela** i livelli preaddestrati: fissa i loro pesi in modo che non cambino durante l'addestramento
- **Sostituisci** l'ultimo livello con uno nuovo, adattato al numero di classi del tuo problema

In questo modo addestri solo il nuovo livello, con molti meno dati e molto meno tempo che se partissi da zero.

In [ ]:
import torch.nn as nn
from torchvision import models

NUEVO_NUMERO_DE_CLASES = 10  # ajusta este valor a tu propio problema

modelo_transferencia = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Congela todas las capas preentrenadas: no se actualizan durante el entrenamiento
for parametro in modelo_transferencia.parameters():
    parametro.requires_grad = False

# Sustituye la última capa lineal (fc) por una nueva, sin congelar
caracteristicas_entrada = modelo_transferencia.fc.in_features
modelo_transferencia.fc = nn.Linear(caracteristicas_entrada, NUEVO_NUMERO_DE_CLASES)

parametros_totales = sum(p.numel() for p in modelo_transferencia.parameters())
parametros_entrenables = sum(p.numel() for p in modelo_transferencia.parameters() if p.requires_grad)

print(f"Parámetros totales:     {parametros_totales:,}")
print(f"Parámetros entrenables: {parametros_entrenables:,}")

# Comprueba que la nueva capa produce la forma de salida esperada
entrada_ejemplo = torch.randn(1, 3, 224, 224)
modelo_transferencia.eval()
with torch.no_grad():
    salida_ejemplo = modelo_transferencia(entrada_ejemplo)
print("Forma de salida:", salida_ejemplo.shape)

## Riepilogo della lezione

- Impilare troppi livelli senza controllo provoca il problema della degradazione: il gradiente svanisce e la rete smette di ottimizzarsi bene
- La connessione residua riformula ogni blocco perché apprenda un residuo, y = F(x) + x, il che garantisce una via diretta per il gradiente
- Una CNN preaddestrata, come ResNet-18 su ImageNet, ti permette di classificare immagini reali senza addestrare nulla da zero
- Il trasferimento di apprendimento congela i livelli preaddestrati e sostituisce l'ultimo livello per adattare il modello al tuo numero di classi

**Apri ora il notebook di pratica di accompagnamento, `../notebooks_practica/clase03_practica_modelos_avanzados_vision.ipynb`, e metti in pratica ciò che hai appena visto.**